# Coalition-Graph Neural Architecture — Experiment

Comparing three architectures on compositional arithmetic:
1. **Dense** — standard transformer FFN
2. **MoE** — top-2 Mixture-of-Experts with flat gating
3. **Coalition** — graph-based seed activation + neighbor recruitment

In [ ]:
# ===== Setup =====
# Run this cell first. If on Colab, it clones the repo and installs deps.

import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in str(globals().get('get_ipython', lambda: ''))

if IN_COLAB:
    !pip install -q torch pyyaml matplotlib seaborn networkx scikit-learn numpy
    # Clone your repo (update URL if needed)
    if not os.path.exists('Dynamic-Coalition-Network-DCN-'):
        !git clone https://github.com/tanushappapogu-max/Dynamic-Coalition-Network-DCN-.git
    os.chdir('Dynamic-Coalition-Network-DCN-')
    print('Repo cloned and ready.')
else:
    # Running locally — make sure you're in the project root
    pass

import sys
sys.path.insert(0, '.')

import torch
print(f'PyTorch: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ===== Optional: Mount Google Drive for saving results =====

SAVE_TO_DRIVE = False  # Set True to save checkpoints to Drive

if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/DCN_results'
    os.makedirs(SAVE_DIR, exist_ok=True)
else:
    SAVE_DIR = 'results'
    os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# ===== Load Config & Generate Data =====

import yaml
from torch.utils.data import DataLoader
from src.data.arithmetic import create_datasets, VOCAB_SIZE, PAD_IDX

with open('configs/experiment.yaml') as f:
    config = yaml.safe_load(f)

print('Generating datasets...')
datasets = create_datasets(config)
for name, ds in datasets.items():
    print(f'  {name}: {len(ds)} examples')

# Show a few examples
for i in range(5):
    expr, result = datasets['train'].data[i]
    print(f'  Example: {expr} = {result}')

batch_size = config['training']['batch_size']
train_loader = DataLoader(datasets['train'], batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(datasets['val'], batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(datasets['test'], batch_size=batch_size, shuffle=False, num_workers=0)
gen_loader = DataLoader(datasets['gen_test'], batch_size=batch_size, shuffle=False, num_workers=0)

In [ ]:
# ===== Train Model A: Dense Baseline =====

from src.models.dense import DenseModel
from src.training.trainer import Trainer

dense_model = DenseModel(config)
dense_trainer = Trainer(dense_model, config, device, save_dir=f'{SAVE_DIR}/checkpoints')
dense_history = dense_trainer.train(train_loader, val_loader)

In [ ]:
# ===== Train Model B: MoE =====

from src.models.moe import MoEModel

moe_model = MoEModel(config)
moe_trainer = Trainer(moe_model, config, device, save_dir=f'{SAVE_DIR}/checkpoints')
moe_history = moe_trainer.train(train_loader, val_loader)

In [ ]:
# ===== Train Model C: Coalition-Graph =====

from src.models.coalition import CoalitionModel

coalition_model = CoalitionModel(config)
coalition_trainer = Trainer(coalition_model, config, device, save_dir=f'{SAVE_DIR}/checkpoints')
coalition_history = coalition_trainer.train(train_loader, val_loader)

In [ ]:
# ===== Training Curves =====

from src.evaluation.visualize import plot_training_curves

histories = {
    'Dense': dense_history,
    'MoE': moe_history,
    'Coalition': coalition_history,
}
plot_training_curves(histories, save_path=f'{SAVE_DIR}/training_curves.png')

In [ ]:
# ===== Evaluate All Three Models =====

from src.evaluation.metrics import evaluate_accuracy, measure_inference_speed, compute_active_params

all_results = {}

for name, model in [('Dense', dense_model), ('MoE', moe_model), ('Coalition', coalition_model)]:
    print(f'\nEvaluating {name}...')
    
    test_acc = evaluate_accuracy(model, test_loader, device)
    gen_acc = evaluate_accuracy(model, gen_loader, device)
    speed = measure_inference_speed(model, test_loader, device, n_samples=500)
    active = compute_active_params(model, test_loader, device)
    
    all_results[name] = {
        'test_accuracy': test_acc['accuracy'],
        'gen_accuracy': gen_acc['accuracy'],
        'inference_ms': speed['mean_ms'],
        'active_params': active['active_params_est'],
        'total_params': active['total_params'],
        'specialization': 'N/A',
    }
    
    print(f'  Test Accuracy:  {test_acc["accuracy"]:.1%}')
    print(f'  Gen Accuracy:   {gen_acc["accuracy"]:.1%}')
    print(f'  Inference:      {speed["mean_ms"]:.2f} ms (mean)')
    print(f'  Active Params:  {active["active_params_est"]:,} / {active["total_params"]:,}')

In [ ]:
# ===== Coalition Specialization Analysis =====

from src.evaluation.metrics import collect_activation_patterns
from src.evaluation.visualize import (
    plot_activation_heatmap,
    plot_graph_structure,
    plot_coalition_size_distribution,
    compute_cluster_purity,
)
import numpy as np

print('Collecting activation patterns...')
patterns = collect_activation_patterns(coalition_model, datasets['test'], device, n_samples=1000)

# Activation heatmap
for layer_idx in range(config['model']['n_layers']):
    plot_activation_heatmap(patterns, layer_idx=layer_idx, save_path=f'{SAVE_DIR}/heatmap_layer{layer_idx+1}.png')

# Coalition size distribution
plot_coalition_size_distribution(patterns, save_path=f'{SAVE_DIR}/coalition_sizes.png')

# Graph structure
edge_weights = None
for layer in coalition_model.layers:
    if hasattr(layer.ffn, 'edge_logits'):
        edge_weights = torch.sigmoid(layer.ffn.edge_logits).detach().cpu().numpy()
        break

if edge_weights is not None:
    plot_graph_structure(patterns, edge_weights, save_path=f'{SAVE_DIR}/graph_structure.png')

# Cluster purity
purity = compute_cluster_purity(patterns, n_clusters=config['evaluation']['n_clusters'])
print(f'\nCluster Purity: {purity["purity"]:.3f}')
if 'clusters' in purity:
    for c, info in purity['clusters'].items():
        print(f'  Cluster {c}: {info["dominant_type"]} (purity={info["purity"]:.2f}, size={info["size"]})')

# Update results
spec_label = 'Strong' if purity['purity'] > 0.6 else 'Moderate' if purity['purity'] > 0.4 else 'Weak'
all_results['Coalition']['specialization'] = f'{spec_label} ({purity["purity"]:.2f})'

In [ ]:
# ===== Final Comparison Table =====

from src.evaluation.visualize import plot_comparison_table

plot_comparison_table(all_results, save_path=f'{SAVE_DIR}/comparison_table.png')

print('\n' + '='*70)
print('FINAL RESULTS')
print('='*70)
print(f'{"Model":<12} {"Test Acc":>10} {"Gen Acc":>10} {"Speed (ms)":>12} {"Active Params":>14} {"Specialization":>16}')
print('-'*70)
for name, r in all_results.items():
    print(f'{name:<12} {r["test_accuracy"]:>9.1%} {r["gen_accuracy"]:>9.1%} {r["inference_ms"]:>11.2f} {r["active_params"]:>13,} {r["specialization"]:>16}')
print('='*70)

In [ ]:
# ===== Save All Results =====

import json

with open(f'{SAVE_DIR}/results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print(f'All results saved to {SAVE_DIR}/')
print('Files:')
for fname in os.listdir(SAVE_DIR):
    if not fname.startswith('.'):
        print(f'  {fname}')